# 12 — Employee Skills Table
Confirm no file has per-employee current skills. Build a controlled placeholder table.

In [1]:

import pandas as pd
import numpy as np
import os

PROC = r'../data/processed'
RAW = r'../data/raw'

ea = pd.read_csv(f'{PROC}/employee_attrition_processed.csv')
ee = pd.read_csv(f'{PROC}/engagement_processed.csv')
ess = pd.read_csv(f'{PROC}/essential_skills_processed.csv')
sw = pd.read_csv(f'{PROC}/software_skills_processed.csv')
print("Files loaded.")


Files loaded.


In [2]:

# ── Confirm: no per-employee skill columns in any file ──
print("=== Checking for per-employee skill data ===")
skill_keywords = ['skill', 'competenc', 'certif', 'python', 'java', 'sql', 'excel', 'tool']

for name, df in [('employee_attrition', ea), ('employee_engagement', ee)]:
    skill_cols = [c for c in df.columns 
                  if any(kw in c.lower() for kw in skill_keywords)]
    print(f"  {name}: skill-like columns = {skill_cols}")

print("\nCONFIRMED: Neither employee file contains per-employee skill records.")
print("The O*NET files (essential_skills, software_skills) define role-level required skills,")
print("NOT individual employee current skills.")
print("\nConclusion: A real deployment would integrate with an LMS/HRIS.")
print("For this build: create a controlled placeholder table.")


=== Checking for per-employee skill data ===
  employee_attrition: skill-like columns = []
  employee_engagement: skill-like columns = []

CONFIRMED: Neither employee file contains per-employee skill records.
The O*NET files (essential_skills, software_skills) define role-level required skills,
NOT individual employee current skills.

Conclusion: A real deployment would integrate with an LMS/HRIS.
For this build: create a controlled placeholder table.


In [3]:

# ── Build placeholder employee skills table ──
# Strategy: assign each employee a subset of skills from their role's O*NET skills.
# We seed with employee_id so results are reproducible.
# Each employee is assigned 30-60% of their role's required skills (mimicking partial skill coverage).

import random
from pathlib import Path

mapping_df = pd.read_csv(f'{PROC}/role_mapping.csv')
role_to_soc = dict(zip(mapping_df['JobRole'], mapping_df['SOC_Code']))

# Get unique element names per SOC code (from essential_skills IM)
soc_skills = ess.groupby('O*NET-SOC Code')['Element Name'].apply(list).to_dict()

records = []
for _, row in ea.iterrows():
    emp_id = row['EmployeeID']
    role = row['JobRole']
    soc = role_to_soc.get(role, None)
    
    if soc and soc in soc_skills:
        all_skills = list(set(soc_skills[soc]))
        # Seed RNG per employee for reproducibility
        rng = random.Random(int(str(emp_id).replace('E','').replace('HR','')[:6]))
        coverage_pct = rng.uniform(0.30, 0.60)  # 30-60% of role skills
        n_skills = max(1, int(len(all_skills) * coverage_pct))
        employee_skills = rng.sample(all_skills, min(n_skills, len(all_skills)))
    else:
        employee_skills = ['Communication', 'Problem Solving']  # minimal fallback
    
    for skill in employee_skills:
        records.append({'employee_id': emp_id, 'current_skill': skill})

skills_df = pd.DataFrame(records)
print(f"Employee skills placeholder table: {skills_df.shape}")
print(f"Employees with skills: {skills_df['employee_id'].nunique()}")
print(f"Avg skills per employee: {skills_df.groupby('employee_id').size().mean():.1f}")
print(f"\nSample:\n{skills_df.head(10)}")


Employee skills placeholder table: (1979, 2)
Employees with skills: 500
Avg skills per employee: 4.0

Sample:
   employee_id          current_skill
0            1    Learning Strategies
1            1  Reading Comprehension
2            1                Writing
3            2             Monitoring
4            2    Learning Strategies
5            2               Speaking
6            2                Science
7            2       Active Listening
8            3               Speaking
9            3                Science


In [4]:

skills_df.to_csv(f'{PROC}/employee_skills_placeholder.csv', index=False)
print("Saved: employee_skills_placeholder.csv")
print("\nNote: This is a controlled placeholder.")
print("Real deployment: integrate with LMS/HRIS to pull actual skill records.")


Saved: employee_skills_placeholder.csv

Note: This is a controlled placeholder.
Real deployment: integrate with LMS/HRIS to pull actual skill records.


**Employee skills table built.** Confirmed: no per-employee skill data exists in source files. Placeholder built using 30-60% O*NET role skill coverage per employee, seeded by EmployeeID for reproducibility.